# King County: preprocessing working copy

Cleaning and feature engineering now use small Python objects in a reusable sklearn pipeline. Run from the repository root with the project's `.venv` kernel.

This copy checks the original full-data baseline, not model quality. Training/evaluation and model saving stay in notebooks and are not run here. See [the baseline reference](docs/notebook-baseline.md).

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline

from king_county.cleaning import HouseCleaner
from king_county.features import HouseFeatureEngineer

## Load the original dataset

In [ ]:
# Load the dataset with pandas
kc_data = pd.read_csv("data/King_County_House_prices_dataset.csv")

## Training-only row exclusion

In [ ]:
# Known exclusion in this dataset only; not part of prediction preprocessing.
kc_data = kc_data.drop(index=15856)

## Reusable cleaning and feature pipeline

`HouseCleaner` applies fixed rules without learning values. `HouseFeatureEngineer.fit()` stores a copy of waterfront coordinates; `transform()` creates both distances and the 19 ordered model inputs using those stored locations. Price, ID, date, and any supplied distances are not passed through.

**Baseline comparison only:** we fit on all retained rows here to match the original notebook. For model evaluation, split first and fit this pipeline on training rows only. T08 will connect it to the estimator, evaluate, and export it. Do not use these full-data fitted references for a held-out test.

In [ ]:
preprocessor = Pipeline(
    [
        ("cleaning", HouseCleaner()),
        ("features", HouseFeatureEngineer()),
    ]
)
X_prepared = preprocessor.fit_transform(kc_data)
X_prepared.head()

## Prepared features

The output keeps row labels and returns the same 19 input names in a fixed order. The feature object retains only waterfront coordinates, not complete training records or sale prices. No model is trained or overwritten.

In [ ]:
engineer = preprocessor.named_steps["features"]
print(f"Prepared rows: {len(X_prepared)}; features: {X_prepared.shape[1]}")
print(f"Stored waterfront references: {len(engineer.waterfront_locations_)}")
print(f"Missing prepared values: {X_prepared.isna().sum().sum()}")